In [11]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3
# Z-score matrix
from sklearn.preprocessing import StandardScaler
from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
from ranking_methods import rank_accuracy                                                                                   # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------


parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  split_animal_protocol_by_day, read_data, get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [12]:
best_feat = None   
named_combo = None
dates_to_keep = []
data_folder = "./data"                              
individual_files = glob.glob(data_folder + "/unwrapped_data/**/*.txt", recursive=True)
ranks = {
    "animal_5": 5,
    "animal_12": 8,
    "animal_9": 6,
    "animal_13": 4,
    "animal_15": 1,
    "animal_3": 3,
    "animal_8": 7,
    "animal_11": 2
}
output_folder = './results/test_results_2026_previous_data2'
if dates_to_keep:
    individual_files = [f for f in individual_files if f.split("/")[-2].split(" ")[0] in dates_to_keep]


## Get files for each animals and build concatenated protocol with all days

In [13]:


#usando sum para concatenar os protocolos gera resultado melhor que o last
#Se eu quiser ler os protocolos eu preciso do zt_0_time, neste caso estou usando 20horas.
#Tambem precisariamos definir o labels dict, neste caso o que utilizariamos?
apply_filtering = True
print("Animal ranks")
for animal, rank in ranks.items():
    print(f"{animal}: {rank}")  
print()
animals = [int(k.split("_")[1]) for k in ranks.keys()]

animals_files = get_sorted_animals_files(individual_files, animals)
for k, v in animals_files.items():
    print(f"{k}: {v}")

animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)




Animal ranks
animal_5: 5
animal_12: 8
animal_9: 6
animal_13: 4
animal_15: 1
animal_3: 3
animal_8: 7
animal_11: 2

Animal 3: 18
Animal 5: 16
Animal 8: 18
Animal 9: 19
Animal 11: 20
Animal 12: 20
Animal 13: 20
Animal 15: 19
3: ['./data/unwrapped_data/2025-03-01_15.02.33/animal_3.txt', './data/unwrapped_data/2025-03-02_15.11.11/animal_3.txt', './data/unwrapped_data/2025-03-04_20.22.59/animal_3.txt', './data/unwrapped_data/2025-03-05_14.57.54/animal_3.txt', './data/unwrapped_data/2025-03-07_08.57.45/animal_3.txt', './data/unwrapped_data/2025-03-10_14.31.46/animal_3.txt', './data/unwrapped_data/2025-03-14_11.12.27/animal_3.txt', './data/unwrapped_data/2025-03-14_15.41.35/animal_3.txt', './data/unwrapped_data/2025-03-14_16.22.42/animal_3.txt', './data/unwrapped_data/2025-03-16_18.11.28/animal_3.txt', './data/unwrapped_data/2025-03-17_14.40.35/animal_3.txt', './data/unwrapped_data/2025-03-18_10.13.19/animal_3.txt', './data/unwrapped_data/2025-03-18_18.29.47/animal_3.txt', './data/unwrapped_da

## Generate features

In [14]:
os.makedirs(output_folder, exist_ok=True)


output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)


output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

y = []
for animal in all_features['animal'].tolist():
    y.append(ranks[animal])
    
# Derived additive features
X_scaled =  get_data_scaled(all_features, feature_cols)

all_features.head(20)

Saving features on ./results/test_results_2026_previous_data2/basic_features.csv
Saving temporal features on ./results/test_results_2026_previous_data2/temporal_features.csv
Saving all features on ./results/test_results_2026_previous_data2/all_features.csv


,animal,total_activity,mean_activity,std_activity,max_activity,min_activity,cv_activity,median_activity,max_median_ratio,activity_per_hour,...,night_ibi_cv,night_bout_len_mean,night_bout_len_cv,night_transitions_per_hour,night_onset_latency_h,night_first_2h_frac,night_gini,night_peak_hour,night_activity_per_bout,night_day_intensity_ratio
animal_3,animal_3,1236.0,0.279955,0.550371,4.800000,-1.200000,1.965929,0.0,4.800000e+09,51.500000,...,7.263770,0.638889,0.599701,0.782963,9.00,-0.012403,3.547242,5.0,1.343798,0.409731
animal_5,animal_5,1070.0,0.242356,0.475873,3.714286,-1.028571,1.963532,0.0,3.714286e+09,44.583333,...,6.881482,0.711340,0.618770,0.703217,7.75,-0.032168,8.058770,6.0,0.574774,0.225111
animal_8,animal_8,2099.0,0.475425,0.926645,9.600000,-1.028571,1.949089,0.0,9.600000e+09,87.458333,...,6.996509,0.693467,0.641862,0.721341,9.25,-0.015265,4.237123,5.0,1.135120,0.222627
animal_9,animal_9,2046.0,0.463420,0.846840,7.542857,-1.285714,1.827370,0.0,7.542857e+09,85.250000,...,7.356063,0.624434,0.721260,0.801087,8.75,-0.022697,5.945514,5.0,0.710002,0.149274
animal_11,animal_11,2904.0,0.657758,1.085883,8.571429,-1.371429,1.650886,0.0,8.571429e+09,121.000000,...,7.701093,0.570248,0.776352,0.877209,9.25,0.000000,0.000000,5.0,-0.038312,-0.005464
animal_12,animal_12,2585.0,0.585504,1.119429,12.342857,-1.285714,1.911906,0.0,1.234286e+10,107.708333,...,6.958017,0.693467,0.697452,0.721341,7.00,-0.057842,14.091819,5.0,0.284565,0.039593
animal_13,animal_13,4497.0,1.018573,1.488940,9.600000,-1.457143,1.461790,0.2,4.800000e+01,187.375000,...,8.007230,0.518797,0.674079,0.964205,8.75,-0.052173,11.955016,1.0,0.296142,0.026835
animal_15,animal_15,2251.0,0.509853,0.991756,8.542857,-1.542857,1.945181,0.0,8.542857e+09,93.791667,...,7.216274,0.676471,0.689625,0.739465,8.25,0.000000,0.000000,5.0,-0.118063,-0.018909


In [15]:
output_correlation = f'{output_folder}/feature_correlations.csv'

corr_df = calculate_correlations(all_features, feature_cols, y, output_path=output_correlation)

# ── Build TOP_FEATURES dynamically from corr_df (cell 4 output) ─────────────
min_rho  = 0.20   # lowered threshold — cast wider net, search will still rank by |ρ|
top_n    = 30     # more candidates

top_corr = corr_df[corr_df.index.isin(feature_cols)].copy()
top_corr = top_corr[top_corr['correlation'].abs() >= min_rho].head(top_n)

TOP_FEATURES = {
    feat: (int(np.sign(row['correlation'])), round(abs(row['correlation']), 3))
    for feat, row in top_corr.iterrows()
}

print(f"TOP_FEATURES built from corr_df ({len(TOP_FEATURES)} features, |ρ| >= {min_rho}):")
print(f"{'Feature':<25} {'sign':>5}  {'|ρ|':>6}  {'p':>7}")
print("-" * 55)
for feat, (sign, rho_abs) in TOP_FEATURES.items():
    p = corr_df.loc[feat, 'p_value']
    sig = "**" if p < 0.01 else " *" if p < 0.05 else " ~" if p < 0.1 else ""
    print(f"  {feat:<23} {sign:>+5}  {rho_abs:>6.3f}  {p:>7.3f} {sig}")

# ── Exhaustive search ────────────────────────────────────────────────────────
available_feats = {f: v for f, v in TOP_FEATURES.items() if f in feature_cols}
feat_names = list(available_feats.keys())
y_true_cmp = y

feat_vectors = {}
for feat, (sign, weight) in available_feats.items():
    col_idx = feature_cols.index(feat)
    feat_vectors[feat] = sign * X_scaled[:, col_idx]

best_results = []
total_combos = sum(len(list(combinations(feat_names, k))) for k in range(1, 6)) * 2

for k in range(1, 6):
    for combo in combinations(feat_names, k):
        composite_c = sum(feat_vectors[f] for f in combo)
        for polarity in [1, -1]:
            pred = rankdata(-polarity * composite_c).astype(int)
            rho_c, p_c = spearmanr(y_true_cmp, pred)
            mae_c = np.abs(y_true_cmp - pred).mean()
            if rho_c > 0:
                best_results.append((rho_c, p_c, mae_c, k, combo, polarity))

best_results.sort(key=lambda x: (-x[0], x[2]))

print(f"\nTop 20 feature combinations (out of {len(best_results)} valid, {total_combos} tested):\n")
print(f"{'ρ':>6} {'p':>7} {'MAE':>5}  {'k':>2}  Features")
print("-" * 90)
for rho_c, p_c, mae_c, k, combo, pol in best_results[:20]:
    sig = "**" if p_c < 0.01 else " *" if p_c < 0.05 else " ~" if p_c < 0.1 else "  "
    print(f"{rho_c:+.3f} {p_c:7.3f} {mae_c:5.2f}  {k:2d}  {', '.join(combo)} {sig}")

# ── Feature vote across top-10 results ──────────────────────────────────────
top_feats_pool = {}
for rho_c, p_c, mae_c, k, combo, pol in best_results[:10]:
    for f in combo:
        top_feats_pool[f] = top_feats_pool.get(f, 0) + rho_c

top_feats_ranked = sorted(top_feats_pool.items(), key=lambda x: -x[1])
print(f"\nMost frequent features in top-10 combos (by accumulated ρ):")
for feat, score in top_feats_ranked:
    sign, _ = available_feats[feat]
    print(f"  {feat:<25s}  score={score:.3f}  sign={'↑' if sign>0 else '↓'}")

# ── Apply the best combination ──────────────────────────────────────────────
best_rho, best_p, best_mae, best_k, best_combo, best_pol = best_results[0]
best_combo_str = " + ".join(best_combo)

print(f"\n→ Best combo (k={best_k}): {list(best_combo)}")
print(f"  ρ={best_rho:+.3f}, p={best_p:.3f}, MAE={best_mae:.2f}")

composite_best = best_pol * sum(feat_vectors[f] for f in best_combo)
pred_best = rankdata(-composite_best).astype(int)
all_features['predicted_rank_best'] = pred_best
all_features['composite_score_best'] = composite_best

comparison_best = all_features.set_index('animal')[['predicted_rank_best', 'composite_score_best']].copy()
comparison_best['true_rank'] = pd.Series(ranks)
comparison_best = comparison_best.sort_values('true_rank')
comparison_best['error'] = (comparison_best['predicted_rank_best'] - comparison_best['true_rank']).abs()

n = len(comparison_best)
print(f"\n  Spearman ρ      : {best_rho:+.3f}  (p={best_p:.3f})")
print(f"  MAE             : {best_mae:.2f} ranks")
print(f"  Exact match     : {(comparison_best['error']==0).sum()}/{n}  ({100*(comparison_best['error']==0).mean():.0f}%)")
print(f"  Within ±1 rank  : {(comparison_best['error']<=1).sum()}/{n}  ({100*(comparison_best['error']<=1).mean():.0f}%)")
print(f"  Within ±2 ranks : {(comparison_best['error']<=2).sum()}/{n}  ({100*(comparison_best['error']<=2).mean():.0f}%)")
print(f"\n{'Animal':<12} {'True':>6} {'Pred':>6} {'Error':>6}")
print("-" * 34)
for animal, row in comparison_best.iterrows():
    flag = " ✓" if row['error'] == 0 else (f" ~{int(row['error'])}" if row['error'] <= 2 else f" ✗{int(row['error'])}")
    print(f"{animal:<12} {int(row['true_rank']):>6} {int(row['predicted_rank_best']):>6} {int(row['error']):>6}{flag}")

# ── Save ranked output sorted by predicted rank ──────────────────────────────
ranked_output = comparison_best.copy().reset_index()
ranked_output = ranked_output.rename(columns={
    'predicted_rank_best': 'predicted_rank',
    'composite_score_best': 'composite_score',
    'error': 'absolute_error',
})
ranked_output['features_used'] = best_combo_str
ranked_output['spearman_rho']  = round(best_rho, 4)
ranked_output['p_value']       = round(best_p, 4)
ranked_output['mae']           = round(best_mae, 4)
ranked_output = ranked_output.sort_values('predicted_rank')

ranked_output_path = f"{output_folder}/best_combo_ranked.csv"
ranked_output.to_csv(ranked_output_path, index=False)

print(f"\n── Best combo ranked output (sorted by predicted rank) ──")
print(ranked_output[['animal', 'predicted_rank', 'true_rank', 'absolute_error', 'composite_score']].to_string(index=False))
print(f"\nSaved to {ranked_output_path}")


Saving correlation on ./results/test_results_2026_previous_data/feature_correlations.csv
TOP_FEATURES built from corr_df (30 features, |ρ| >= 0.2):
Feature                    sign     |ρ|        p
-------------------------------------------------------
  cosinor_amplitude          +1   0.833    0.010  *
  power_24h                  +1   0.833    0.010  *
  rhythm_ratio_24_over_harm    +1   0.786    0.021  *
  night_gini                 +1   0.755    0.031  *
  night_first_2h_frac        -1   0.755    0.031  *
  interdaily_stability       +1   0.595    0.120 
  longest_active_run         +1   0.591    0.123 
  min_activity               +1   0.551    0.157 
  daily_peak_std             -1   0.548    0.160 
  power_12h                  -1   0.548    0.160 
  relative_amplitude         +1   0.500    0.207 
  peak_to_mean_ratio         +1   0.500    0.207 
  ibi_p90                    +1   0.482    0.226 
  short_gap_frac             +1   0.476    0.233 
  day_frac                   -1   0

In [15]:
if 'y_original' not in globals():
    y_original = np.asarray(y).copy()
    all_features_original = all_features.copy()
    X_scaled_original = X_scaled.copy()

shuffle_seed = 12
rng = np.random.default_rng(shuffle_seed)
shuffle_idx = rng.permutation(len(y_original))

y = y_original[shuffle_idx].tolist()
all_features = all_features_original.iloc[shuffle_idx].reset_index(drop=True)
X_scaled = X_scaled_original[shuffle_idx]

print(f"Applied random shuffle with seed={shuffle_seed}")
print("Shuffle indices:", shuffle_idx.tolist())
print("Animals after shuffle:", all_features['animal'].tolist())
print("Ranks after shuffle:", y)




Applied random shuffle with seed=12
Shuffle indices: [1, 6, 3, 5, 2, 7, 4, 0]
Animals after shuffle: ['animal_5', 'animal_13', 'animal_9', 'animal_12', 'animal_8', 'animal_15', 'animal_11', 'animal_3']
Ranks after shuffle: [5, 4, 6, 8, 7, 1, 2, 3]


In [16]:
feature_rhos_path = "./data/feature_rhos_new.csv"
feature_rhos_df = pd.read_csv(feature_rhos_path)

if {'feature', 'rho'}.issubset(feature_rhos_df.columns):
    feature_rhos_new = feature_rhos_df.set_index('feature')['rho']
else:
    raise ValueError(
        "`feature_rhos_previous.csv` must contain `feature` and `rho` columns. "
        "Recreate it with: feature_rhos.to_frame('rho').rename_axis('feature').to_csv(...)"
    )

feature_rhos_new.head()

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [ ]:
# best_feat = None
# named_combo = None
# #best_feat = 'rhythm_ratio_24_over_harm'
# #named_combo = ['power_24h', 'high_activity_frac', 'activity_per_bout']
# if best_feat is None:
#     print("Best feat is none, computing")
#     best_feat = corr_df[corr_df.index.isin(feature_cols)]['correlation'].abs().idxmax()

# if named_combo is None:
#     print("Named combo is none, using computed")
#     named_combo = list(best_combo) if 'best_combo' in dir() else None
    

# # named_combo = ['power_24h', 'high_activity_frac', 'activity_per_bout']
best_feat = 'cosinor_amplitude'
named_combo = ['power_24h', 'short_gap_frac', 'night_ibi_cv']

best_feat = 'rhythm_ratio_24_over_harm'
named_combo = ['power_24h', 'high_activity_frac', 'activity_per_bout']

feature_rhos, proxies_raw = build_all_proxies(
    all_features, feature_cols, X_scaled, y,
    k=3, best_feat_idx=best_feat,
    named_combo=named_combo,
    feature_rhos=feature_rhos_new
)

feature_rhos_path = "./data/feature_rhos_previous.csv"
feature_rhos.to_frame('rho').rename_axis('feature').to_csv(feature_rhos_path)
print(f"Saved `feature_rhos` with feature names to {feature_rhos_path}")

print(f"Best single feature: {best_feat}")
print('Top feature correlations (Spearman):')
print(feature_rhos.head(10))

proxy_output_path = f"{output_folder}/proxy_summary.csv"
proxy_rows, proxy_summary, r = evaluate_proxies(proxies_raw, all_features, y, verbose=True, output_path=proxy_output_path)




Using feature_rhos provided externally (e.g. from a reference dataset).
Saved `feature_rhos` with feature names to ./data/feature_rhos_previous.csv
Best single feature: cosinor_amplitude
Top feature correlations (Spearman):
feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
median_activity             -0.646554
activity_kurtosis           -0.608392
high_activity_frac           0.577552
activity_skew               -0.559441
bout_len_cv                 -0.545455
Name: rho, dtype: float64

Best feature (cosinor_amplitude):
  Predicted ranks are generated with: rankdata(scores_oriented, method="ordinal")
  Metrics: rho=0.833  tau=0.643  mae=1.00  acc=37.5%
  Animals in DataFrame row order: ['animal_5', 'animal_13', 'animal_9', 'animal_12', 'animal_8', 'animal_15', 'animal_11', 'animal_3']
  True ranks aligned to DataFrame rows: [5, 4, 6, 8, 7, 1

In [20]:
for name,score in proxies_raw.items():
    print(score)
    ord_idx = np.argsort(score)
    print(ord_idx)
    ordered_animals = all_features.iloc[ord_idx]['animal'].tolist()
    print(name)
    pred_rank = rankdata(score, method='ordinal')
    print(ordered_animals)
    print(pred_rank)
    print(y)

    metrics = rank_accuracy(y, pred_rank)
    print(metrics)
    print()

[0.28166364 0.30777037 0.33153019 0.32993751 0.35518579 0.0922626
 0.24644688 0.31213783]
[5 6 0 1 7 3 2 4]
Best feature (cosinor_amplitude)
['animal_15', 'animal_11', 'animal_5', 'animal_13', 'animal_3', 'animal_12', 'animal_9', 'animal_8']
[3 4 7 6 8 1 2 5]
[5, 4, 6, 8, 7, 1, 2, 3]
{'accuracy': 0.375, 'within_1': 0.625, 'within_2': 1.0, 'mae': 1.0, 'rho': 0.8333333333333335}

[ 0.25127916 -0.15607532  1.09579365  2.47029338  1.98637152 -2.84678643
 -1.41587769 -1.38499826]
[5 6 7 1 0 2 4 3]
Best combo (power_24h + short_gap_frac + night_ibi_cv)
['animal_15', 'animal_11', 'animal_3', 'animal_13', 'animal_5', 'animal_9', 'animal_8', 'animal_12']
[5 4 6 8 7 1 2 3]
[5, 4, 6, 8, 7, 1, 2, 3]
{'accuracy': 1.0, 'within_1': 1.0, 'within_2': 1.0, 'mae': 0.0, 'rho': 1.0}

[ 0.08375972 -0.05202511  0.36526455  0.82343113  0.66212384 -0.94892881
 -0.47195923 -0.46166609]
[5 6 7 1 0 2 4 3]
Combo sign-aligned mean (power_24h + short_gap_frac + night_ibi_cv)
['animal_15', 'animal_11', 'animal_3', 'a